# Description Scraper

In [ ]:
# Scrap Again For Description
import pandas as pd
import requests
from bs4 import BeautifulSoup
import time
import random

FILENAME = "../data/compiled_no_desc2.csv"      # Change file names accordingly <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
df = pd.read_csv(FILENAME)   

def scrape_job_detail(url):
    try:
        headers = {
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
            "Accept-Language": "en-US,en;q=0.9",
        }
        resp = requests.get(url, headers=headers, timeout=15)
        soup = BeautifulSoup(resp.text, "html.parser")

        # Lokasi Kerja
        location_div = soup.find("div", id="job-locations")
        if location_div:
            items = location_div.find_all("li")
            location = "; ".join(li.get_text(strip=True) for li in items) if items else location_div.get_text(strip=True)
        else:
            location = ""

        # Penerangan Kerja
        desc_div = soup.find("div", id="job-description")
        if desc_div:
            description = desc_div.get_text(separator="\n", strip=True)
        else:
            description = ""
        
        if not location and not description:
            return "Job Closed", "Job Closed"

        return location, description

    except Exception as e:
        print(f"Error scraping {url}: {e}")
        return "Error", "Error"

# Force string dtype from the start
if "scraped_location" not in df.columns:
    df["scraped_location"] = pd.array([""] * len(df), dtype="string")
if "scraped_description" not in df.columns:
    df["scraped_description"] = pd.array([""] * len(df), dtype="string")

# Also cast if columns already exist but are float
df["scraped_location"] = df["scraped_location"].astype("string").fillna("")
df["scraped_description"] = df["scraped_description"].astype("string").fillna("")

to_scrape = df[
    (df["scraped_location"].fillna("") == "") 
   |(df["scraped_location"] == "Error") 
#    |(df["scraped_location"] == "Job Closed")  # Uncomment to recheck for Description set as Job Closed
].index

print(f"Scraping {len(to_scrape)} jobs...")

for i, idx in enumerate(to_scrape):
    url = df.at[idx, "url"]
    if pd.isna(url) or url in ("", "N/A", "nan"):
        continue

    loc, desc = scrape_job_detail(url)
    df.loc[idx, "scraped_location"] = str(loc)
    df.loc[idx, "scraped_description"] = str(desc)

    # Save every 50 rows in case it crashes
    if i % 50 == 0:
        df.to_csv(FILENAME, index=False, encoding="utf-8-sig")
        print(f"Progress: {i}/{len(to_scrape)} — last: {url}")

    # Random delay to avoid getting blocked
    time.sleep(random.uniform(1.5, 3.5))

df.to_csv(FILENAME, index=False, encoding="utf-8-sig")
print("Done!")